# Demo MedGemma en direct via Streamlit (Colab)

> **Prototype pedagogique. Non destine au diagnostic.**

Ce notebook clone TON depot, charge MedGemma, ecrit l'app Streamlit et ouvre un tunnel public pour la demo.

**Avant de lancer :** `Execution > Modifier le type d'execution > GPU` (T4).
Puis lance les cellules **dans l'ordre**.

## 1. Cloner ton depot + installer + outil de tunnel

In [ ]:
import os
if not os.path.exists('DS-7C-assistant-radiologue-virtuel'):
    !git clone https://github.com/TonneauBenjamin/DS-7C-assistant-radiologue-virtuel.git
%cd /content/DS-7C-assistant-radiologue-virtuel
!pip install -q -U transformers accelerate bitsandbytes sentencepiece streamlit kagglehub
!pip install -q -r medapp/requirements.txt
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!nvidia-smi -L

## 2. Connexion Hugging Face
MedGemma est *gated* : accepte les conditions sur https://huggingface.co/google/medgemma-4b-it, puis colle un token *read*.

In [ ]:
from huggingface_hub import login
login()  # colle ton token hf_...

## 3. (Optionnel) Recuperer 6 vraies radios pour la demo
Pour pouvoir tester sans avoir d'images sur ton PC, on met quelques radios dans `data/real_cases/`. Ignore cette cellule si tu preferes televerser tes propres images.

In [ ]:
try:
    import kagglehub, shutil, random
    from pathlib import Path
    kagglehub.login()  # username + key (kaggle.json)
    base = Path(kagglehub.dataset_download('paultimothymooney/chest-xray-pneumonia'))
    test_root = next(base.rglob('test'))
    random.seed(0)
    dst = Path('data/real_cases'); dst.mkdir(parents=True, exist_ok=True)
    picked = []
    for sub, label in [('NORMAL', 'normal'), ('PNEUMONIA', 'suspected_opacity')]:
        fs = sorted((test_root / sub).glob('*.jpeg')); random.shuffle(fs)
        for i, f in enumerate(fs[:3]):
            out = dst / f'CXR_REAL_{label}_{i}{f.suffix.lower()}'; shutil.copy(f, out); picked.append(out.name)
    print('Images de demo pretes :', picked)
except Exception as e:
    print('Etape optionnelle ignoree :', e)

## 4. Télécharger MedGemma pour l'interface TrueVision (medapp)
L'interface complète (`medapp/`) détecte automatiquement le GPU et propose alors les modèles `medgemma-baseline` et `medgemma-improved` dans son écran d'analyse, en plus du modèle jouet. On télécharge ici les poids (~5 Go) pour que l'app n'ait plus qu'à les charger.

In [ ]:
# Telecharge les poids sur le disque (cache partage avec l'app Streamlit),
# sans charger le modele en GPU ici : c'est l'app qui le chargera dans SON processus.
from huggingface_hub import snapshot_download
from src.medgemma_inference import MODEL_ID, is_available

assert is_available(), "GPU CUDA requis : Execution > Modifier le type d'execution > GPU (T4)"
snapshot_download(MODEL_ID)
print("Poids MedGemma en cache. Premiere analyse dans l'interface : ~1-2 min (chargement GPU), puis rapide.")

## 5. Lancer l'interface TrueVision (medapp) + ouvrir le tunnel
La cellule affiche une adresse **https://....trycloudflare.com** : ouvre-la dans un nouvel onglet, connecte-toi avec `admin@demo.local` / `admin` (mode démo local), puis dans l'écran d'analyse choisis le modèle **medgemma-improved**. **Ne stoppe pas cette cellule** pendant la démo (elle maintient le tunnel).

In [ ]:
import subprocess, time
subprocess.Popen(['streamlit', 'run', 'medapp/app.py',
                  '--server.port', '8501', '--server.headless', 'true'])
time.sleep(12)
print('Ouvre l URL trycloudflare.com qui apparait ci-dessous (login : admin@demo.local / admin) :')
!./cloudflared tunnel --url http://localhost:8501